# OpenCodeReasoning challenge Fine-tuning stage 1 (SFT)

Now that our dataset is ready, we can start with the first stage of fine-tuning. This notebook addresses the requirement of identifying bugs and providing commentary. Simultaneously, this process also allows the model to view questions and the corresponding coding patterns used.

This is a notebook I derived from Unsloth's own Qwen2.5-Coder Kaggle notebook. The original notebook can be found at https://www.kaggle.com/code/danielhanchen/kaggle-qwen-2-5-coder-14b-conversational

## Kaggle is slow - you'll have to wait **5 minutes** for it to install.

I suggest you to use our free Colab notebooks instead. I linked our Llama 3.1 8b Colab notebook here: [notebook](https://colab.research.google.com/drive/1Ys44kVvmeZtnICzWz0xgpRnrIOjZAuxp?usp=sharing)

In [1]:
%%capture
!pip install pip3-autoremove
!pip-autoremove torch torchvision torchaudio -y
!pip uninstall torch torchvision torchaudio
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu118
!pip install unsloth

# Load the base model `Qwen/Qwen2.5-Coder-1.5B` and its tokeniser using Unsloth's FastLanguageModel

Unsloth is a popular resource-efficient fine-tuning library that reduces the time taken for training models significantly, making it especially useful in constrained conditions like the free tier of Kaggle. Using FastLanguageModel, the model is loaded into memory in an efficient manner. Fine-tuning also uses significantly less VRAM, avoiding CUDA OOM errors.

In [2]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 8192 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HUGGINGFACE_TOKEN = user_secrets.get_secret("HF_TOKEN")


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-Coder-1.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    token = HUGGINGFACE_TOKEN, # use one if using gated models like meta-llama/Llama-2-7b-hf
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-02-21 10:22:12.022027: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771669332.271137     687 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771669332.341930     687 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771669332.892152     687 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771669332.892188     687 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771669332.892191     687 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.2.1: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu118. CUDA: 7.5. CUDA Toolkit: 11.8. Triton: 3.3.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.31.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


### Add LoRA Adapters
We now implement LoRA, a parameter-efficient fine-tuning method. LoRA adapters make it so that we only need to update 1 to 10% of all parameters!
- `r` here stands for LoRA Rank. Lower rank results in faster but not very smart training. 16 is chosen as a middle-ground to balance effectiveness and feasibility
- `lora_alpha` is a factor that determines the strength that the adapter has on the overall model. In essence, it affects the overall influence of fine-tuning. `lora_alpha = r` is a standard starting point for this value

In [1]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

NameError: name 'FastLanguageModel' is not defined

<a name="Data"></a>
### Data Prep
We now use the `Qwen-2.5` format for conversation style finetunes. Qwen renders multi turn conversations like below:

```
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
What is 2+2?<|im_end|>
<|im_start|>assistant
It's 4.<|im_end|>

```

To achieve this, we load qwen-2.5's tokenizer using `get_chat_template`. Now we can translate each open-AI compatible messages object to qwen-2.5's template and attach it to a new field `text`. This is done by `formatting_prompts_func`

In [4]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }
pass

## Dataset download and modulation
Now we need to download our dataset from [the Data Curation Notebook](https://www.kaggle.com/code/irishcoffeee/dataset-curation-ocr2-challenge) and modulate it to match the below HuggingFace message format:

```
{
    "conversations":[
        {"role": "system", "content": "You are an assistant"}
        {"role": "user", "content": "What is 2+2?"}
        {"role": "assistant", "content": "It's 4."}
    ]
}
```

In [62]:
from datasets import load_dataset
dataset = load_dataset("santhosh-m/ocr2-wrong-solns-curated", split = "train")

### Adjusting CoT length after noticing unstable behaviour

While working on the third notebook for [GRPO training](https://www.kaggle.com/code/irishcoffeee/leap-la-grpo-finetuning-ocr2-santhoshm), the model showed huge instability due to excessively long Chain-of-Thought (CoT) responses that may have overwhelmed the 1.5B parameter model. This led to repetitions, hallucinations, etc. and subsequently, inefficient training.

As a result, I returned to this step and introduced a filter to retain only samples with CoT length < 5k. This change was necessary to stabilise GRPO training and stay within resource constraints.

In [64]:
max_CoT_length = 5000

dataset = dataset.filter(
    lambda x: len(x["ocr_qwq_critique"]) <= max_CoT_length)

Filter:   0%|          | 0/856 [00:00<?, ? examples/s]

In [65]:
def my_preprocess_function(example):
    system_instr = """You are a Quality Assurance Engineer. Evaluate code formally and rigorously.
For each task, document your reasoning step by step, identify issues, and provide an objective final assessment in a structured format. Avoid informal language or assumptions; base all conclusions on evidence.
Always start with the <CoT>...</CoT> tags and provide your assessment. Follow that with the tags <propposed_fix>...</proposed_fix> with the correct code running python code."""
    prompt = f"""
I am trying to solve this coding question and wrote this code, but there is something wrong with it. Can you find what the problem is?
The question:
{example['taco_question']}

My code:
{example['ocr_user_code']}
"""

    response = f"""
{example['ocr_qwq_critique'].replace('<think>','<CoT>').replace('</think>','</CoT>')}

<proposed_fix>
{example['taco_soln']}
</proposed_fix>
"""

    return {
        'conversations': [
            {
                'role':'system',
                'content':system_instr
            },
            {
                'role':'user',
                'content':prompt
            },
            {
                'role':'assistant',
                'content':response
            }
        ]
    }


Now create the conversation format

In [66]:
dataset = dataset.map(my_preprocess_function, 
                                remove_columns=dataset.features.keys()
                               )

Map:   0%|          | 0/856 [00:00<?, ? examples/s]

In [67]:
# dataset[0]

Now we apply `formatting_prompts_func` to include the 'text' field.

In [68]:
dataset = dataset.map(formatting_prompts_func, batched = True,)

Map:   0%|          | 0/856 [00:00<?, ? examples/s]

In [69]:
dataset[5]['text']

'<|im_start|>system\nYou are a Quality Assurance Engineer. Evaluate code formally and rigorously.\nFor each task, document your reasoning step by step, identify issues, and provide an objective final assessment in a structured format. Avoid informal language or assumptions; base all conclusions on evidence.\nAlways start with the <CoT>...</CoT> tags and provide your assessment. Follow that with the tags <propposed_fix>...</proposed_fix> with the correct code running python code.<|im_end|>\n<|im_start|>user\n\nI am trying to solve this coding question and wrote this code, but there is something wrong with it. Can you find what the problem is?\nThe question:\nGiven an integer x, find 2 integers a and b such that: \n\n  * 1 ≤ a,b ≤ x \n  * b divides a (a is divisible by b). \n  * a ⋅ b>x. \n  * a/b<x. \n\nInput\n\nThe only line contains the integer x (1 ≤ x ≤ 100).\n\nOutput\n\nYou should output two integers a and b, satisfying the given conditions, separated by a space. If no pair of int

Now we may delete all unused variables to clear up RAM if required

In [70]:
import gc

# del <whichever we are not gonna use anymore>

gc.collect()

2828

<a name="Train"></a>
### Train the model
Now let's use Huggingface TRL's `SFTTrainer`! More docs here: [TRL SFT docs](https://huggingface.co/docs/trl/sft_trainer). Unfortunately, in SFT, steps are slow in the Kaggle free tier with T4 GPU. Therefore, we only do 150 steps. Ideally it would be 1-2 epochs.

In [71]:
# Shuffle dataset before training

dataset = dataset.shuffle(seed=42)

In [72]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 150,
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/856 [00:00<?, ? examples/s]

In [73]:
print(len(tokenizer(dataset[5]["text"]).input_ids))


2041


We also use Unsloth's `train_on_responses_only` method to only train on the assistant outputs and ignore the loss on the user's inputs.

In [74]:
# Sanity check - Checking raw tokenised version before and after train_on_response_only
tmp = tokenizer(dataset[5]["text"])
print(len(tmp["input_ids"]))
# print(tokenizer.decode(tmp["input_ids"]))


2041


In [75]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n\n",
    response_part = "<|im_start|>assistant\n\n",
)

Map (num_proc=8):   0%|          | 0/856 [00:00<?, ? examples/s]

Filter (num_proc=8):   0%|          | 0/856 [00:00<?, ? examples/s]

We verify masking is actually done:

In [76]:
len(trainer.train_dataset[5]["input_ids"])


2041

In [77]:
# tokenizer.decode(trainer.train_dataset[5]["input_ids"])

In [78]:
space = tokenizer(" ", add_special_tokens = False).input_ids[0]
tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[5]["labels"]])

'                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               <CoT>\nOkay, let\'s see. The user provided a solution and wants me to check if it\'s correct. Hmm, the problem is to find the missing elements in the range from 0 to the maximum element in the array. The examples show that the output should list ranges of missing numbers separated by spaces, using hyphens for r

We can see the System and User prompts are successfully masked! Training and loss calculations would be evaluated only from the above visible portion of the conversation (only on agent response)

In [79]:
#@title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
12.365 GB of memory reserved.


In [80]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 856 | Num Epochs = 2 | Total steps = 150
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,0.828600
2,0.833100
3,0.775100
4,0.861600
5,0.895300
6,0.853000
7,0.905500
8,0.881500
9,0.861200
10,0.873200


In [81]:
#@title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory         /max_memory*100, 3)
lora_percentage = round(used_memory_for_lora/max_memory*100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

1589.2079 seconds used for training.
26.49 minutes used for training.
Peak reserved memory = 12.365 GB.
Peak reserved memory for training = 0.0 GB.
Peak reserved memory % of max memory = 84.907 %.
Peak reserved memory for training % of max memory = 0.0 %.


<a name="Inference"></a>
### Inference

Let's run the model! You can change the instruction and input - leave the output blank!


In [82]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536, padding_idx=151665)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [83]:
inference = dataset[20]['conversations']

In [84]:
messages = [x for x in inference if x['role'] in ['system','user']]

In [87]:
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 4096, use_cache = True,
                         temperature = 1.5, min_p = 0.1)
tokenizer.batch_decode(outputs)

['<|im_start|>system\nYou are a Quality Assurance Engineer. Evaluate code formally and rigorously.\nFor each task, document your reasoning step by step, identify issues, and provide an objective final assessment in a structured format. Avoid informal language or assumptions; base all conclusions on evidence.\nAlways start with the <CoT>...</CoT> tags and provide your assessment. Follow that with the tags <propposed_fix>...</proposed_fix> with the correct code running python code.<|im_end|>\n<|im_start|>user\n\nI am trying to solve this coding question and wrote this code, but there is something wrong with it. Can you find what the problem is?\nThe question:\nYou are given three integers $a$, $b$, and $c$. Determine if one of them is the sum of the other two.\n\n\n-----Input-----\n\nThe first line contains a single integer $t$ ($1 \\leq t \\leq 9261$) — the number of test cases.\n\nThe description of each test case consists of three integers $a$, $b$, $c$ ($0 \\leq a, b, c \\leq 20$).\n

 You can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

#### Important: Bias from the above inference:
The above inference was done from a data sample used for training. Therefore while it looks decent, it must be noted that it is a sample the model has seen already. Due to the very small number of samples used for training it is possible that the model is not ready yet.

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [88]:
from huggingface_hub import login
# model.save_pretrained("lora_model") # Local saving
# tokenizer.save_pretrained("lora_model")
# login(token=HUGGINGFACE_TOKEN)
model.push_to_hub("santhosh-m/qwen-coder-ocr2-lora_model-v2", token = HUGGINGFACE_TOKEN) # Online saving
tokenizer.push_to_hub("santhosh-m/qwen-coder-ocr2-lora_model-v2", token = HUGGINGFACE_TOKEN) # Online saving

README.md:   0%|          | 0.00/580 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/santhosh-m/qwen-coder-ocr2-lora_model-v2


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

## Save to float16 for VLLM

This is done to make the SFT-finetuned model compatible with VLLM, to boost the next stage of finetuning, which is reinforcement learning with GRPO. The initial idea was to have:
> (Base model) + (SFT LoRA adapter) + (GRPO LoRA adapter)

But stacking two LoRA adapters came with a limitation - both LoRA Adapters would have been of the same parameters. This also denied us the opportunity to use VLLM to boost training speed. Now after merging, we will instead have:
> (Merged Base model with SFT adapter) + (GRPO LoRA adapter)

In [90]:
# Merge to 16bit
if True: model.push_to_hub_merged("santhosh-m/ocr2-sft-lora-merged-v2", tokenizer, save_method = "merged_16bit", token = HUGGINGFACE_TOKEN)

config.json:   0%|          | 0.00/765 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Unsloth: Merging weights into 16bit:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:52<00:00, 52.83s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/santhosh-m/ocr2-sft-lora-merged-v2`


### Updates added in for V2

- Firstly, I filtered the dataset to only include small problems which the model can handle without losing its sanity. I did this by limiting the length of CoT (qwq_critique) down to 5000 characters)
- Second, since GRPO was pretty slow in my first attempt after V1 of this notebook, I decided to save models to be compatible with VLLM. 

# Notebook ends here

The below cells are helpful in case one wishes to save the models in other formats, a residual from the original unsloth notebook. I am leaving them here but commented out for easy use later.

The model that we obtained after fine-tuning and merging will act as the base model for our next stage of fine-tuning. Proceed to the [GRPO training notebook](https://www.kaggle.com/code/irishcoffeee/leap-la-grpo-finetuning-ocr2-santhoshm)

Now if you want to load the LoRA adapters we just saved for inference, uncomment and set `False` to `True`:

In [ ]:
# if False:
#     from unsloth import FastLanguageModel
#     model, tokenizer = FastLanguageModel.from_pretrained(
#         model_name = "lora_model", # YOUR MODEL YOU USED FOR TRAINING
#         max_seq_length = max_seq_length,
#         dtype = dtype,
#         load_in_4bit = load_in_4bit,
#     )
#     FastLanguageModel.for_inference(model) # Enable native 2x faster inference

# messages = [
#     {"role": "user", "content": "Describe a tall tower in the capital of France."},
# ]
# inputs = tokenizer.apply_chat_template(
#     messages,
#     tokenize = True,
#     add_generation_prompt = True, # Must add for generation
#     return_tensors = "pt",
# ).to("cuda")

# from transformers import TextStreamer
# text_streamer = TextStreamer(tokenizer, skip_prompt = True)
# _ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 128,
#                    use_cache = True, temperature = 1.5, min_p = 0.1)

You can also use Hugging Face's `AutoModelForPeftCausalLM`. Only use this if you do not have `unsloth` installed. It can be hopelessly slow, since `4bit` model downloading is not supported, and Unsloth's **inference is 2x faster**.

In [ ]:
# if False:
#     # I highly do NOT suggest - use Unsloth if possible
#     from peft import AutoPeftModelForCausalLM
#     from transformers import AutoTokenizer
#     model = AutoPeftModelForCausalLM.from_pretrained(
#         "lora_model", # YOUR MODEL YOU USED FOR TRAINING
#         load_in_4bit = load_in_4bit,
#     )
#     tokenizer = AutoTokenizer.from_pretrained("lora_model")

#### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# # Merge to 16bit
# if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
# if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# # Merge to 4bit
# if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
# if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# # Just LoRA adapters
# if False: model.save_pretrained_merged("model", tokenizer, save_method = "lora",)
# if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "lora", token = "")

#### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/drive/1WZDi7APtQ9VsvOrQSSC5DDtxq159j8iZ?usp=sharing)

In [ ]:
# # Save to 8bit Q8_0
# if False: model.save_pretrained_gguf("model", tokenizer,)
# # Remember to go to https://huggingface.co/settings/tokens for a token!
# # And change hf to your username!
# if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

# # Save to 16bit GGUF
# if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
# if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

# # Save to q4_k_m GGUF
# if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
# if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")

# # Save to multiple GGUF options - much faster if you want multiple!
# if False:
#     model.push_to_hub_gguf(
#         "hf/model", # Change hf to your username!
#         tokenizer,
#         quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
#         token = "", # Get a token at https://huggingface.co/settings/tokens
#     )

Now, use the `model-unsloth.gguf` file or `model-unsloth-Q4_K_M.gguf` file in `llama.cpp` or a UI based system like `GPT4All`. You can install GPT4All by going [here](https://gpt4all.io/index.html).

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/u54VK8m8tk) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Zephyr DPO 2x faster [free Colab](https://colab.research.google.com/drive/15vttTpzzVXv_tJwEk-hIcQ0S9FcEWvwP?usp=sharing)
2. Llama 7b 2x faster [free Colab](https://colab.research.google.com/drive/1lBzz5KeZJKXjvivbYvmGarix9Ao6Wxe5?usp=sharing)
3. TinyLlama 4x faster full Alpaca 52K in 1 hour [free Colab](https://colab.research.google.com/drive/1AZghoNBQaMDgWJpi4RbffGM1h6raLUj9?usp=sharing)
4. CodeLlama 34b 2x faster [A100 on Colab](https://colab.research.google.com/drive/1y7A0AxE3y8gdj4AVkl2aZX47Xu3P1wJT?usp=sharing)
5. Mistral 7b [free Kaggle version](https://www.kaggle.com/code/danielhanchen/kaggle-mistral-7b-unsloth-notebook)
6. We also did a [blog](https://huggingface.co/blog/unsloth-trl) with 🤗 HuggingFace, and we're in the TRL [docs](https://huggingface.co/docs/trl/main/en/sft_trainer#accelerate-fine-tuning-2x-using-unsloth)!
7. `ChatML` for ShareGPT datasets, [conversational notebook](https://colab.research.google.com/drive/1Aau3lgPzeZKQ-98h69CCu1UJcvIBLmy2?usp=sharing)
8. Text completions like novel writing [notebook](https://colab.research.google.com/drive/1ef-tab5bhkvWmBOObepl1WgJvfvSzn5Q?usp=sharing)
9. We make Phi-3 Medium / Mini **2x faster**! See our [Phi-3 Medium notebook](https://colab.research.google.com/drive/1hhdhBa1j_hsymiW9m-WzxQtgqTH_NHqi?usp=sharing)
10. We make Gemma-2 9b / 27b **2x faster**! See our [Gemma-2 9b notebook](https://colab.research.google.com/drive/1vIrqH5uYDQwsJ4-OO3DErvuv4pBgVwk4?usp=sharing)
11. To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/drive/1WZDi7APtQ9VsvOrQSSC5DDtxq159j8iZ?usp=sharing)
12. We make Mistral NeMo 12B 2x faster and fit in under 12GB of VRAM! [Mistral NeMo notebook](https://colab.research.google.com/drive/17d3U-CAIwzmbDRqbZ9NnpHxCkmXB6LZ0?usp=sharing)
13. [**NEW**] We make Llama 3.2 1/3B 2x faster! [Llama 3.2 notebook](https://colab.research.google.com/drive/1T5-zKWM_5OD21QHwXHiV9ixTRR7k3iB9?usp=sharing)
13. [**NEW**] We fixed a major bug in gradient accumulation. [Blog post](https://unsloth.ai/blog/gradient)

<div class="align-center">
  <a href="https://github.com/unslothai/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/u54VK8m8tk"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://ko-fi.com/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Kofi button.png" width="145"></a></a> Support our work if you can! Thanks!
</div>